In [1]:
%pip install datasets matplotlib scikit-learn
from datasets import load_dataset

ds = load_dataset("rajistics/indian_food_images")


Note: you may need to restart the kernel to use updated packages.


/Users/muskangoyal/miniforge3/envs/food_env_fix/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


In [3]:
ds=ds["train"].train_test_split(test_size=0.2,seed=42)
val_test=ds["test"].train_test_split(test_size=0.5, seed=42)

train_ds=ds["train"]
val_ds=val_test["train"]
test_ds=val_test["test"]

In [4]:
IMG_SIZE = 224
BATCH_SIZE=32

def preprocess(example):
    image=example["image"]
    image=image.resize((IMG_SIZE, IMG_SIZE))
    image=np.array(image, stype=np.float32)/255.0
    label=example["label"]
    return image, label


In [5]:
def to_tf_dataset(hf_ds, shuffle=True, is_training=True):
    def generator():
        for example in hf_ds:
            img = example["image"]
            # Ensure image is in RGB (converts RGBA or Grayscale automatically)
            img = img.convert("RGB") 
            yield np.array(img), example["label"]

    tf_ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            # We use None for height/width because images might be different sizes
            tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8), 
            tf.TensorSpec(shape=(), dtype=tf.int64),
        ),
    )

    def preprocess(image, label):
        # Resize and scale
        image = tf.image.resize(image, (224, 224))
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    tf_ds = tf_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        tf_ds = tf_ds.shuffle(1000)
    
    if is_training:
        tf_ds = tf_ds.repeat()

    return tf_ds.batch(32).prefetch(tf.data.AUTOTUNE)

# Re-initialize your datasets
train_tf = to_tf_dataset(train_ds, shuffle=True, is_training=True)
val_tf   = to_tf_dataset(val_ds, shuffle=False, is_training=False) # No repeat
test_tf  = to_tf_dataset(test_ds, shuffle=False, is_training=False) # No repeat

# Calculate steps accurately so the progress bar moves
steps_per_epoch = len(train_ds) // 32
val_steps = len(val_ds) // 32

2026-03-18 12:03:25.481770: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-18 12:03:25.481803: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-18 12:03:25.481811: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-03-18 12:03:25.481841: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-18 12:03:25.481851: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [6]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2)
])


In [7]:
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False


In [8]:
num_classes = len(set(train_ds["label"]))


In [9]:
inputs = tf.keras.Input(shape=(224,224,3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)

outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)


In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [11]:
history = model.fit(
    train_tf,
    validation_data=val_tf,
    epochs=15,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps
)


Epoch 1/15


2026-03-18 12:03:27.153113: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


133/133 ━━━━━━━━━━━━━━━━━━━━ 50s 298ms/step - accuracy: 0.1034 - loss: 3.7198 - val_accuracy: 0.3105 - val_loss: 2.4059
Epoch 2/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 38s 286ms/step - accuracy: 0.1814 - loss: 3.1689 - val_accuracy: 0.4688 - val_loss: 1.9177
Epoch 3/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 44s 335ms/step - accuracy: 0.2751 - loss: 2.7262 - val_accuracy: 0.5566 - val_loss: 1.5766
Epoch 4/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 41s 307ms/step - accuracy: 0.3466 - loss: 2.3884 - val_accuracy: 0.6309 - val_loss: 1.3319
Epoch 5/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 41s 309ms/step - accuracy: 0.4124 - loss: 2.1309 - val_accuracy: 0.6680 - val_loss: 1.1750
Epoch 6/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 41s 312ms/step - accuracy: 0.4572 - loss: 1.9488 - val_accuracy: 0.7012 - val_loss: 1.0574
Epoch 7/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 44s 331ms/step - accuracy: 0.4866 - loss: 1.8358 - val_accuracy: 0.7207 - val_loss: 1.0052
Epoch 8/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 48s 361ms/step - accuracy: 0.5296 - loss: 1.6578 - val

In [12]:
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in test_tf:
    preds = model.predict(images)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred))


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
              precision    recall  f1-score   support

           0       0.83      1.00      0.91        24
           1       0.69      0.81      0.75        27
           2       0.86      0.91      0.88        46
           3       0.75      0.52      0.62        23
           4       0.64      0.91      0.75        32
           5      

2026-03-18 12:14:37.389820: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [13]:
model.save("indian_food_classifier.h5")


In [14]:
import json
from datasets import load_dataset

dataset = load_dataset("rajistics/indian_food_images")

# Get label names
label_names = dataset["train"].features["label"].names

# Save to json
with open("label_map.json", "w") as f:
    json.dump(label_names, f)

print("label_map.json created ✅")


label_map.json created ✅
